In [1]:

# TEAM FIT 

import pandas as pd
import numpy as np

# Load master rankings
master = pd.read_csv("data/outputs/master_rankings.csv")

# Tier 1 players
tier1 = master[master['tier'] == 'Tier 1'].copy()

print(f"Total players in master: {len(master)}")
print(f"Tier 1 players to score: {len(tier1)}")


Total players in master: 220
Tier 1 players to score: 149


In [2]:
# -- TEAM FIT WEIGHTS --

FIT_WEIGHT_DEFENSE    = 0.40  
FIT_WEIGHT_POSITION   = 0.20  
FIT_WEIGHT_OFFENSE    = 0.20  
FIT_WEIGHT_COMPETITION = 0.10 
FIT_WEIGHT_3PT        = 0.10  # 

total = (FIT_WEIGHT_DEFENSE + FIT_WEIGHT_POSITION + 
         FIT_WEIGHT_OFFENSE + FIT_WEIGHT_COMPETITION + 
         FIT_WEIGHT_3PT)

print("Team Fit Weights:")
print(f"  Defensive fit:     {FIT_WEIGHT_DEFENSE*100:.0f}%")
print(f"  Positional need:   {FIT_WEIGHT_POSITION*100:.0f}%")
print(f"  Offensive fit:     {FIT_WEIGHT_OFFENSE*100:.0f}%")
print(f"  Competition:       {FIT_WEIGHT_COMPETITION*100:.0f}%")
print(f"  3PT tendency:      {FIT_WEIGHT_3PT*100:.0f}%")
print(f"  Total:             {total*100:.0f}%")


Team Fit Weights:
  Defensive fit:     40%
  Positional need:   20%
  Offensive fit:     20%
  Competition:       10%
  3PT tendency:      10%
  Total:             100%


In [3]:
#POSITIONAL NEED SCORES 


POSITIONAL_NEED_SCORES = {
    'PG': 1,   # Covered - Edmead, Whitty
    'SG': 1,   # Covered - Hammond, Adams
    'SF': 4,   # High - McNeil and Keene - Depth Needed
    'PF': 2,   # Moderate - Yalaho, Wilkins
    'C':  5,   # Critical - Evans - Backup/Depth Needed
}

# Normalize to 0-100 scale
max_need = max(POSITIONAL_NEED_SCORES.values())
POSITIONAL_NEED_NORMALIZED = {
    pos: (need / max_need) * 100 
    for pos, need in POSITIONAL_NEED_SCORES.items()
}

print("Positional Need Scores (0-100):")
for pos, score in POSITIONAL_NEED_NORMALIZED.items():
    print(f"  {pos}: {score:.0f}/100")


Positional Need Scores (0-100):
  PG: 20/100
  SG: 20/100
  SF: 80/100
  PF: 40/100
  C: 100/100


In [4]:
# TEAM FIT SCORING FUNCTION 
from sklearn.preprocessing import MinMaxScaler

fit = tier1.copy()

# Normalize DBPR, OBPR, and competition to 0-100
scaler = MinMaxScaler(feature_range=(0, 100))

fit['defense_score']     = scaler.fit_transform(fit[['DBPR']])
fit['offense_score']     = scaler.fit_transform(fit[['OBPR']])
fit['competition_score'] = scaler.fit_transform(fit[['Adj_Team_Eff_Margin']])

# 3PT tendency — use OBPR as proxy since we don't have
# direct 3PT data 
# system suggests 3PT compatibility
fit['three_pt_score'] = fit['offense_score']

# Positional need score — look up each player's position
fit['need_score'] = fit['Position'].map(POSITIONAL_NEED_NORMALIZED)

# Handle any positions not in our map
fit['need_score'] = fit['need_score'].fillna(20)

# Calculate final fit score
fit['fit_score'] = (
    fit['defense_score']     * FIT_WEIGHT_DEFENSE +
    fit['need_score']        * FIT_WEIGHT_POSITION +
    fit['offense_score']     * FIT_WEIGHT_OFFENSE +
    fit['competition_score'] * FIT_WEIGHT_COMPETITION +
    fit['three_pt_score']    * FIT_WEIGHT_3PT
)

print(f"Fit score range: {fit['fit_score'].min():.1f} - {fit['fit_score'].max():.1f}")
print(f"Average fit score: {fit['fit_score'].mean():.1f}")


Fit score range: 16.1 - 81.7
Average fit score: 43.4


In [5]:
# TOP 25 BY TEAM FIT SCORE

top_fit = fit.sort_values('fit_score', ascending=False).head(25)

print("=== TOP 25 NC STATE FIT SCORES ===")
print(f"{'Rank':<5} {'Name':<22} {'Pos':<5} {'Team':<22} {'Fit':<8} {'Defense':<9} {'Offense':<9} {'Need'}")
print("-" * 85)

for i, (_, row) in enumerate(top_fit.iterrows(), 1):
    print(f"{i:<5} {row['Name']:<22} {row['Position']:<5} {str(row['Team']):<22} {row['fit_score']:<8.1f} {row['defense_score']:<9.1f} {row['offense_score']:<9.1f} {row['need_score']:.0f}")


=== TOP 25 NC STATE FIT SCORES ===
Rank  Name                   Pos   Team                   Fit      Defense   Offense   Need
-------------------------------------------------------------------------------------
1     RJ Godfrey             SF    Clemson                81.7     88.0      73.8      80
2     Ernest Udeh Jr.        C     Miami (Fla.)           81.5     89.2      58.9      100
3     Milan Momcilovic       PF    Iowa State             78.9     77.2      100.0     40
4     Micah Handlogten       C     Florida                77.2     74.3      63.0      100
5     Baba Miller            PF    Cincinnati             77.0     100.0     72.2      40
6     Tyler Nickel           SF    Vanderbilt             73.9     55.4      89.3      80
7     Allen Graves           PF    Santa Clara            72.4     71.4      95.2      40
8     Tounde Yessoufou       SF    Baylor                 71.8     70.2      70.3      80
9     Tyler Harris           SF    Vanderbilt             71.4   

In [6]:
# FIT SCORES BY POSITION 

positions = ['PG', 'SG', 'SF', 'PF', 'C']

for pos in positions:
    pos_players = fit[fit['Position'] == pos].sort_values(
        'fit_score', ascending=False).head(10)
    
    if len(pos_players) == 0:
        continue
    
    print(f"\n=== TOP 10 {pos} BY FIT SCORE ===")
    print(f"{'Rank':<5} {'Name':<22} {'Team':<22} {'Fit':<8} {'Defense':<9} {'Offense'}")
    print("-" * 70)
    
    for i, (_, row) in enumerate(pos_players.iterrows(), 1):
        print(f"{i:<5} {row['Name']:<22} {str(row['Team']):<22} {row['fit_score']:<8.1f} {row['defense_score']:<9.1f} {row['offense_score']:.1f}")



=== TOP 10 PG BY FIT SCORE ===
Rank  Name                   Team                   Fit      Defense   Offense
----------------------------------------------------------------------
1     Skyy Clark             UCLA                   63.3     67.1      81.7
2     Jayden Pierre          TCU                    60.2     72.8      64.8
3     Trey Campbell          Northern Iowa          60.1     78.8      60.5
4     Jordan Pope            Texas                  55.3     48.4      80.5
5     Seth Trimble           North Carolina         54.2     54.9      70.0
6     Kenyon Giles           Wichita State          53.6     55.1      70.4
7     Derek Simpson          Saint Joseph's         53.5     61.5      66.2
8     Tre Holloman           NC State               52.1     54.3      64.0
9     Desmond Claude         Washington             50.5     53.1      60.9
10    Isaiah Watts           Maryland               49.9     64.7      47.4

=== TOP 10 SG BY FIT SCORE ===
Rank  Name                

In [7]:
# Tier 2 and Tier 3

tier2_fit = master[master['tier'] == 'Tier 2'].copy()
tier3_fit = master[master['tier'] == 'Tier 3 - Development'].copy()

# Apply positional need score to both tiers
tier2_fit['need_score'] = tier2_fit['Position'].map(POSITIONAL_NEED_NORMALIZED).fillna(20)
tier3_fit['need_score'] = tier3_fit['Position'].map(POSITIONAL_NEED_NORMALIZED).fillna(20)

# No full fit score possible without metrics
tier2_fit['fit_score'] = None
tier3_fit['fit_score'] = None

print("=== TIER 2 WITH POSITIONAL NEED ===")
print(tier2_fit[['Name', 'Position', 'Year', 'composite_score', 'need_score']].sort_values(
    'need_score', ascending=False).head(10).to_string())

print("\n=== TIER 3 WITH POSITIONAL NEED ===")
print(tier3_fit[['Name', 'Position', 'Year', 'Height', 'Weight', 'need_score']].sort_values(
    'need_score', ascending=False).head(10).to_string())


=== TIER 2 WITH POSITIONAL NEED ===
                       Name Position   Year  composite_score  need_score
153        Ja'Quay Randolph        C     JR             89.0       100.0
152             Ryan Blount       SF     SO             90.0        80.0
169            Jahki Howard       SF     SO             87.0        80.0
149           Quadir Martin       PF     SO             92.0        40.0
163  Marqus Mitrovic Marion       PF  RS-SO             88.0        40.0
157        Anthony Robinson       PF     JR             89.0        40.0
166        Antonio Pusateri       PF     JR             88.0        40.0
164              Jevon Hill       PF     SR             88.0        40.0
172           Omar Adegbola       SG  RS-SO             85.0        20.0
171        Travonne Jackson       SG  RS-JR             85.0        20.0

=== TIER 3 WITH POSITIONAL NEED ===
                  Name Position   Year Height  Weight  need_score
174  Vincent Iwuchukwu        C     SR   6-11   220.0     

In [8]:

# Combine all three tiers
full_output = pd.concat([fit, tier2_fit, tier3_fit], ignore_index=True)

# Save complete file
full_output.to_csv("data/outputs/full_rankings_with_fit.csv", index=False)

print(f"Tier 1: {len(fit)}")
print(f"Tier 2: {len(tier2_fit)}")
print(f"Tier 3: {len(tier3_fit)}")
print(f"Total:  {len(full_output)}")


Tier 1: 149
Tier 2: 25
Tier 3: 46
Total:  220
